# Notebook 2: YOLOv8 Training
**ANPR System – MIPA UGM Parking Lot**

Trains three YOLOv8 variants (n/s/m) for license plate detection:
- 100 epochs, batch 16, imgsz 640
- Adam optimizer, cosine LR schedule
- Saves best weights for each variant
- Compares training curves side-by-side

## 1. Setup

In [ ]:
!pip install -q ultralytics
from google.colab import drive
drive.mount('/content/drive')

import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

DRIVE_ROOT = '/content/drive/MyDrive/ANPR_MIPA_UGM'
DATA_YAML = f'{DRIVE_ROOT}/processed/data.yaml'
WEIGHTS_DIR = f'{DRIVE_ROOT}/weights'

## 2. Training Configuration

In [ ]:
TRAIN_ARGS = {
    'data': DATA_YAML,
    'epochs': 100,
    'batch': 16,
    'imgsz': 640,
    'optimizer': 'Adam',
    'lr0': 0.001,
    'cos_lr': True,
    'patience': 20,
    'save': True,
    'device': 0 if torch.cuda.is_available() else 'cpu',
    'workers': 4,
    'cache': True,
    'augment': True,
    'plots': True,
}

VARIANTS = ['yolov8n', 'yolov8s', 'yolov8m']
print('Training configuration ready.')

## 3. Train YOLOv8n (Nano)

In [ ]:
from ultralytics import YOLO
import shutil, os

model_n = YOLO('yolov8n.pt')
results_n = model_n.train(
    project=f'{DRIVE_ROOT}/runs',
    name='yolov8n_plates',
    **TRAIN_ARGS
)
best_n = f'{DRIVE_ROOT}/runs/yolov8n_plates/weights/best.pt'
print(f'YOLOv8n best: {best_n}')

## 4. Train YOLOv8s (Small)

In [ ]:
model_s = YOLO('yolov8s.pt')
results_s = model_s.train(
    project=f'{DRIVE_ROOT}/runs',
    name='yolov8s_plates',
    **TRAIN_ARGS
)
best_s = f'{DRIVE_ROOT}/runs/yolov8s_plates/weights/best.pt'
print(f'YOLOv8s best: {best_s}')

## 5. Train YOLOv8m (Medium)

In [ ]:
model_m = YOLO('yolov8m.pt')
results_m = model_m.train(
    project=f'{DRIVE_ROOT}/runs',
    name='yolov8m_plates',
    **TRAIN_ARGS
)
best_m = f'{DRIVE_ROOT}/runs/yolov8m_plates/weights/best.pt'
print(f'YOLOv8m best: {best_m}')

## 6. Compare Training Curves

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

runs = {
    'YOLOv8n': f'{DRIVE_ROOT}/runs/yolov8n_plates/results.csv',
    'YOLOv8s': f'{DRIVE_ROOT}/runs/yolov8s_plates/results.csv',
    'YOLOv8m': f'{DRIVE_ROOT}/runs/yolov8m_plates/results.csv',
}

metrics_to_plot = [
    ('train/box_loss', 'Training Box Loss'),
    ('val/box_loss', 'Validation Box Loss'),
    ('metrics/mAP50(B)', 'mAP@0.5'),
    ('metrics/mAP50-95(B)', 'mAP@0.5:0.95'),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
colors = {'YOLOv8n': '#4c72b0', 'YOLOv8s': '#55a868', 'YOLOv8m': '#dd8452'}

for ax, (col, title) in zip(axes.flat, metrics_to_plot):
    for name, csv_path in runs.items():
        try:
            df = pd.read_csv(csv_path)
            df.columns = df.columns.str.strip()
            if col in df.columns:
                ax.plot(df['epoch'], df[col], label=name, color=colors[name], linewidth=2)
        except Exception as e:
            print(f'Could not load {csv_path}: {e}')
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.legend()
    ax.grid(alpha=0.3)

plt.suptitle('YOLOv8 Training Comparison (n / s / m)', fontsize=14)
plt.tight_layout()
plt.savefig(f'{DRIVE_ROOT}/training_curves.png', dpi=150)
plt.show()
print('Curves saved.')

## 7. Copy Best Weights to Drive

In [ ]:
import os, shutil

os.makedirs(WEIGHTS_DIR, exist_ok=True)
weight_map = {
    'best_yolov8n.pt': best_n,
    'best_yolov8s.pt': best_s,
    'best_yolov8m.pt': best_m,
}
for dest_name, src_path in weight_map.items():
    dest = f'{WEIGHTS_DIR}/{dest_name}'
    if os.path.exists(src_path):
        shutil.copy2(src_path, dest)
        size_mb = os.path.getsize(dest) / 1e6
        print(f'Saved {dest_name} ({size_mb:.1f} MB)')
    else:
        print(f'NOT FOUND: {src_path}')